# 04 — EDA and Feature Engineering: LGD Model

This notebook covers the exploratory data analysis and feature engineering for the LGD model features. All analysis is performed exclusively on the defaulted loan population (~128k observations). Outliers are treated and the cleaned dataframe is saved for training.

**Input:** `data/processed/checkpoint_DataFrame_for_EDA.pkl`, `data/processed/features_LGD_for_EDA.pkl`  
**Output:** `data/processed/checkpoint_df_LGD_training.pkl`, `data/processed/features_LGD_num_for_training.pkl`, `data/processed/features_LGD_cat_for_training.pkl`

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os

from feature_plot import plot_boxplots_list, plot_categoricals
from feature_statistics import dataset_audit, outlier_summary
from clean_features_LGD import clean_features_LGD

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_DataFrame_for_EDA.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

with open("data/processed/features_LGD_for_EDA.pkl", "rb") as f:
    features_LGD = pickle.load(f)

# LGD model trains exclusively on defaulted loans
df_LGD = df_master_features.loc[
    df_master_features["Target_Variable_Default"] == 1,
    ["Target_Variable_Loss"] + features_LGD
].copy()

print(f"df_LGD shape: {df_LGD.shape}")
print(f"Features: {features_LGD}")

## 2. Initial Audit

In [ ]:
audit_LGD = dataset_audit(df_LGD[features_LGD])
audit_LGD

## 3. Numeric Features

### 3.1 Distribution Overview

In [ ]:
# loan_term_months is binary categorical — excluded from numeric analysis
num_features_LGD = [
    f for f in df_LGD[features_LGD].select_dtypes(include=[np.number]).columns.tolist()
    if f != "loan_term_months"
]

plot_boxplots_list(df_LGD, num_features_LGD, cols=4)

### 3.2 Outlier Summary

In [ ]:
outlier_summary_LGD = outlier_summary(df_LGD, num_features_LGD)
outlier_summary_LGD

### 3.3 Feature-level Analysis

#### `num_open_credit_lines`

Extreme outlier percentage of 0.33% with maximum of 74. Value counts above upper 1.5 fence confirm gradual decreasing distribution up to ~63, followed by isolated observations at 68 and 74. Discontinuity in upper tail — capping at p99 justified.

In [ ]:
df_LGD.loc[df_LGD["num_open_credit_lines"] > 25.50, "num_open_credit_lines"].value_counts().sort_index()

#### `num_rev_trades_op_in_24mths`

Extreme outlier percentage of 0.25% with maximum of 49. Value counts above upper fence confirm a smooth, monotonically decreasing distribution — no treatment required.

In [ ]:
df_LGD.loc[df_LGD["num_rev_trades_op_in_24mths"] > 11.00, "num_rev_trades_op_in_24mths"].value_counts().sort_index()

#### `total_credit_revolving_bal`

Skewness of 8.57. Log1p transformation was explored and discarded: post-transformation skewness of -2.81 indicates the transformation inverts the asymmetry due to the distribution shape. Capping at p99 without log1p is the final treatment — consistent with `max_bal_owed` in the PD model. See `docs/lgd_feature_treatment.md` for full justification.

In [ ]:
# Verify zero count — 711 zeros (0.55%), insufficient to explain skew inversion
print(f"Zero count: {(df_LGD['total_credit_revolving_bal'] == 0).sum()}")
print(f"Skew positive values only: {df_LGD[df_LGD['total_credit_revolving_bal'] > 0]['total_credit_revolving_bal'].skew():.4f}")

In [ ]:
# Cross-reference with annual_income for economic coherence
df_LGD[df_LGD["total_credit_revolving_bal"] > df_LGD["total_credit_revolving_bal"].quantile(0.99)][
    ["annual_income", "total_credit_revolving_bal"]
].describe()

### 3.4 Apply Treatments

Detailed treatment decisions are documented in `docs/lgd_feature_treatment.md`. Summary:

| Feature | Treatment |
|---|---|
| `total_credit_revolving_bal` | Cap p99 (no log1p — distribution pathology) |
| `monthly_payment` | None |
| `num_open_credit_lines` | Cap p99 (discontinuous upper tail) |
| `loan_term_months` | Encoding only (binary categorical) |
| `num_rev_trades_op_in_24mths` | None |
| `used_credit_share` | Cap at 100 (economic ceiling) |
| `interest_rate` | None |
| `annual_income` | Cap p99, log1p if skew > 2 post-capping |
| `emp_length` | None |
| `dept_paym_income_ratio` | None (no negatives in defaulted population) |

In [ ]:
df_LGD_num_clean = clean_features_LGD(df_LGD)

In [ ]:
# Post-treatment audit
audit_LGD_num_clean = dataset_audit(df_LGD_num_clean[num_features_LGD])
audit_LGD_num_clean

In [ ]:
# Post-treatment boxplots
plot_boxplots_list(df_LGD_num_clean, num_features_LGD, cols=4)

In [ ]:
# Post-treatment outlier summary
outlier_summary_LGD_num_clean = outlier_summary(df_LGD_num_clean, num_features_LGD)
outlier_summary_LGD_num_clean

## 4. Categorical Features

### `loan_term_months`

Binary categorical variable (36 or 60 months). No NaNs, no outliers possible. One-Hot Encoding at training stage.

In [ ]:
cat_features_LGD = ["loan_term_months"]

print(df_LGD_num_clean["loan_term_months"].value_counts(normalize=True))
plot_categoricals(df_LGD_num_clean, cat_features_LGD)

In [ ]:
df_LGD_clean = df_LGD_num_clean.copy()

## 5. Checkpoint: Save LGD Training Dataframe

In [ ]:
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/checkpoint_df_LGD_training.pkl", "wb") as f:
    pickle.dump({"df_LGD_clean": df_LGD_clean}, f)

with open("data/processed/features_LGD_num_for_training.pkl", "wb") as f:
    pickle.dump(num_features_LGD, f)

with open("data/processed/features_LGD_cat_for_training.pkl", "wb") as f:
    pickle.dump(cat_features_LGD, f)

print(f"Saved. Shape: {df_LGD_clean.shape}")

In [ ]:
# Load checkpoint (skip re-running)
with open("data/processed/checkpoint_df_LGD_training.pkl", "rb") as f:
    data = pickle.load(f)
df_LGD_clean = data["df_LGD_clean"]

with open("data/processed/features_LGD_num_for_training.pkl", "rb") as f:
    num_features_LGD = pickle.load(f)

with open("data/processed/features_LGD_cat_for_training.pkl", "rb") as f:
    cat_features_LGD = pickle.load(f)